# Get better at a job by doing it

**The job.** Fetch a record from one of several sources, tidy it, check it.
Some sources are better than others. Nobody has told us which.

Every other notebook picks a route from numbers somebody wrote down when the
graph was drawn. Those numbers are guesses. This one throws them away and uses
what actually happened.

The loop is four steps and they are all real here:

1. run the plan,
2. keep the receipt,
3. fold the receipt into evidence,
4. let the next search start from what is known.

**In:** a job with three ways to do each of three steps.
**Out:** a route that reaches the best one there is, and the numbers to prove it.

In [1]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

ready


## The job

Three steps, three ways to do each. Nine nodes, twenty-seven routes — small
enough to check the answer by hand at the end, which is the point of using a
small example for this.

In [2]:
from browsergraph.evidence import Evidence, stages_of
from browsergraph.workbench import OptimizationObjective, OptimizationProfile
from browsergraph import search

SOURCES = ["fetch.alpha", "fetch.beta", "fetch.gamma"]
TIDIERS = ["tidy.strict", "tidy.loose", "tidy.smart"]
CHECKS  = ["check.shallow", "check.deep", "check.paranoid"]

nodes = (
    [node(n, "fetch", [], [("out", "Record")], runtime={"deterministic": False})
     for n in SOURCES]
    + [node(n, "tidy", [("in", "Record")], [("out", "Record")]) for n in TIDIERS]
    + [node(n, "check", [("in", "Record")], [("out", "Verdict")]) for n in CHECKS]
)

stages = [
    stage("fetch", "Fetch a record", [],                  [("out", "Record")],  "fetch", SOURCES),
    stage("tidy",  "Tidy it",        [("in", "Record")],  [("out", "Record")],  "tidy",  TIDIERS),
    stage("check", "Check it",       [("in", "Record")],  [("out", "Verdict")], "check", CHECKS),
]

bench = build("Fetch, tidy, check",
              "Get one good record, using whichever combination actually works.",
              stages, nodes,
              [Edge("fetch", "tidy"), Edge("tidy", "check")])

bench = replace(bench, optimization_profiles=(
    OptimizationProfile(id="p.quality", name="Quality first", objectives=(
        OptimizationObjective("quality", "maximize", 1.0),)),))

print("routes:", bench.route_count())

problems: none
routes: 27


In [3]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 226" width="1100" height="226" style="max-width:none" role="img"><defs><marker id="bg55319252-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Fetch a record</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">3 candidates</text></g><text x="479.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="386" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="395" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Tidy it</text><text x="395" y="108.0" font-size="9.5" fill="#68737f">3 candidates</text></g><text x="805.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="712" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="721" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Check it</text><text x="721" y="108.0" font-size="9.5" fill="#68737f">3 candidates</text></g><path d="M246,100.0 C316.0,100.0 316.0,100.0 386,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg55319252-arrow)"/><path d="M572,100.0 C642.0,100.0 642.0,100.0 712,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg55319252-arrow)"/></svg>', title='Fetch, tidy, check — shape', note='a chain. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=226)

## The truth, which the graph does not know

Below is how the world actually behaves. The notebook uses it only to *decide
whether a run succeeded* — it never tells the graph, the search, or the scorer.
That is the whole experiment: can the system work this out from outcomes alone?

`fetch.gamma` is the good source. `tidy.smart` is the good tidier. And the
checks barely matter, which is a real thing that happens and is worth seeing
the system discover rather than assume.

In [4]:
import random

TRUTH = {
    "fetch.alpha": 0.35, "fetch.beta": 0.55, "fetch.gamma": 0.90,
    "tidy.strict": 0.55, "tidy.loose": 0.60, "tidy.smart": 0.88,
    "check.shallow": 0.80, "check.deep": 0.82, "check.paranoid": 0.78,
}
best_route = {"fetch": "fetch.gamma", "tidy": "tidy.smart", "check": "check.deep"}

def true_quality(route):
    """A route works only if every step works. The product, not the average."""
    out = 1.0
    for candidate in route.values():
        out *= TRUTH[candidate]
    return out

print(f"{'route':<48}{'true quality':>13}")
print(f"{'the best one there is':<48}{true_quality(best_route):>13.3f}")
worst = {"fetch": "fetch.alpha", "tidy": "tidy.strict", "check": "check.paranoid"}
print(f"{'the worst one there is':<48}{true_quality(worst):>13.3f}")

route                                            true quality
the best one there is                                   0.649
the worst one there is                                  0.150


## The functions

Each node succeeds or fails according to the truth above. Nothing else about it
is visible to the rest of the notebook.

In [5]:
rng = random.Random(20260810)

def make(candidate):
    def run_step(**kw):
        if rng.random() > TRUTH[candidate]:
            raise RuntimeError(f"{candidate} failed this time")
        return {"by": candidate}
    return run_step

runtime = execute.Runtime({c: make(c) for c in TRUTH})
print(f"{len(TRUTH)} functions, one per candidate")

9 functions, one per candidate


## Run it fifty times, learning as we go

Each pass: search inside a small budget using what is known so far, compile,
run, take the receipt, fold it in. Nothing else.

In [6]:
store = Evidence()
history = []

for run_index in range(120):
    found = search.within(bench, bench.optimization_profiles[0],
                          evaluations=40, evidence=store, seed=run_index)
    plan = compile_route(bench, found.route)
    result = execute.run(plan, runtime, strict=False)

    # The loop, in one line: what just happened becomes what is known.
    store.from_receipt(result.receipt(task=f"run-{run_index}"))

    history.append({"run": run_index + 1, "route": dict(found.route),
                    "ok": result.ok, "true": true_quality(found.route)})

worked = sum(1 for h in history if h["ok"])
print(f"{worked} of {len(history)} runs succeeded end to end")

65 of 120 runs succeeded end to end


## Did the choice get better?

The honest measure is the *true* quality of the route it picked — the thing the
system cannot see. Success rate alone is noisy at this sample size.

In [7]:
def block(rows):
    return sum(h["true"] for h in rows) / len(rows)

print(f"{'runs':<18}{'true quality of the route chosen':>34}")
for start in range(0, len(history), 20):
    window = history[start:start + 20]
    print(f"{f'{start + 1}-{start + len(window)}':<18}{block(window):>34.3f}")
print()
print(f"{'the best possible':<18}{true_quality(best_route):>34.3f}")
print(f"{'picking at random':<18}"
      f"{sum(TRUTH[c] for c in SOURCES)/3 * sum(TRUTH[c] for c in TIDIERS)/3 * sum(TRUTH[c] for c in CHECKS)/3:>34.3f}")

runs                true quality of the route chosen
1-20                                           0.334
21-40                                          0.473
41-60                                          0.618
61-80                                          0.595
81-100                                         0.577
101-120                                        0.618

the best possible                              0.649
picking at random                              0.325


The same numbers as a picture, which is where the shape of it shows.

The two dashed lines are what make it readable. A rising line on its own proves
nothing — it could be rising towards mediocre. Against a ceiling and a floor it
says something: *how far it got, and how much was left*.

In [8]:
random_pick = (sum(TRUTH[c] for c in SOURCES) / 3
               * sum(TRUTH[c] for c in TIDIERS) / 3
               * sum(TRUTH[c] for c in CHECKS) / 3)

viz.trend([h["true"] for h in history], smooth=10,
          title="true quality of the route it chose, run by run",
          label="the loop never sees this number — it only sees pass or fail",
          reference={"best possible": true_quality(best_route),
                     "picking at random": random_pick})

Figure(svg='<svg viewBox="0 0 940 300" width="940" height="300" style="max-width:none" role="img"><line x1="62" y1="46" x2="62" y2="256" stroke="#dfe5ec" stroke-width="1"/><line x1="62" y1="256" x2="790" y2="256" stroke="#dfe5ec" stroke-width="1"/><text x="54" y="243.2" text-anchor="end" font-size="10" fill="#68737f">0.16</text><text x="54" y="66.80000000000001" text-anchor="end" font-size="10" fill="#68737f">0.643</text><text x="62" y="280" font-size="10" fill="#68737f">1</text><text x="790" y="280" text-anchor="end" font-size="10" fill="#68737f">120</text><line x1="62" y1="60.5" x2="790" y2="60.5" stroke="#1f8a4c" stroke-width="1.2" stroke-dasharray="5 4" opacity=".75"/><text x="798" y="64.5" font-size="10" fill="#1f8a4c">best possible</text><line x1="62" y1="179.1" x2="790" y2="179.1" stroke="#c98a2b" stroke-width="1.2" stroke-dasharray="5 4" opacity=".75"/><text x="798" y="183.1" font-size="10" fill="#c98a2b">picking at random</text><polyline points="62.0,241.5 68.1,198.9 74.2,72.1 80.4,241.5 86.5,198.9 92.6,72.1 98.7,241.5 104.8,198.9 110.9,72.1 117.1,241.5 123.2,72.1 129.3,241.5 135.4,72.1 141.5,198.9 147.6,198.9 153.8,241.5 159.9,241.5 166.0,198.9 172.1,198.9 178.2,72.1 184.4,241.5 190.5,198.9 196.6,72.1 202.7,241.5 208.8,241.5 214.9,198.9 221.1,72.1 227.2,72.1 233.3,241.5 239.4,198.9 245.5,72.1 251.6,72.1 257.8,72.1 263.9,72.1 270.0,72.1 276.1,72.1 282.2,72.1 288.4,72.1 294.5,72.1 300.6,72.1 306.7,72.1 312.8,72.1 318.9,72.1 325.1,72.1 331.2,72.1 337.3,72.1 343.4,72.1 349.5,72.1 355.6,72.1 361.8,72.1 367.9,72.1 374.0,72.1 380.1,72.1 386.2,72.1 392.4,72.1 398.5,72.1 404.6,72.1 410.7,72.1 416.8,72.1 422.9,72.1 429.1,72.1 435.2,72.1 441.3,72.1 447.4,72.1 453.5,72.1 459.6,72.1 465.8,72.1 471.9,241.5 478.0,72.1 484.1,72.1 490.2,72.1 496.4,72.1 502.5,72.1 508.6,72.1 514.7,72.1 520.8,72.1 526.9,72.1 533.1,72.1 539.2,72.1 545.3,72.1 551.4,72.1 557.5,72.1 563.6,72.1 569.8,72.1 575.9,72.1 582.0,72.1 588.1,198.9 594.2,72.1 600.4,241.5 606.5,72.1 612.6,72.1 618.7,72.1 624.8,72.1 630.9,72.1 637.1,72.1 643.2,72.1 649.3,72.1 655.4,72.1 661.5,72.1 667.6,72.1 673.8,72.1 679.9,72.1 686.0,72.1 692.1,72.1 698.2,72.1 704.4,72.1 710.5,72.1 716.6,72.1 722.7,72.1 728.8,72.1 734.9,72.1 741.1,72.1 747.2,72.1 753.3,72.1 759.4,72.1 765.5,72.1 771.6,72.1 777.8,72.1 783.9,72.1 790.0,72.1" fill="none" stroke="#2d6cb5" stroke-width="1.2" opacity=".35"/><polyline points="62.0,241.5 68.1,220.2 74.2,170.8 80.4,188.5 86.5,190.6 92.6,170.8 98.7,180.9 104.8,183.2 110.9,170.8 117.1,177.9 123.2,161.0 129.3,165.2 135.4,165.2 141.5,161.0 147.6,161.0 153.8,177.9 159.9,177.9 166.0,177.9 172.1,190.6 178.2,173.6 184.4,190.6 190.5,186.3 196.6,186.3 202.7,190.6 208.8,194.8 214.9,190.6 221.1,173.6 227.2,161.0 233.3,165.2 239.4,177.9 245.5,161.0 251.6,148.3 257.8,148.3 263.9,131.3 270.0,114.4 276.1,101.7 282.2,101.7 288.4,101.7 294.5,84.7 300.6,72.1 306.7,72.1 312.8,72.1 318.9,72.1 325.1,72.1 331.2,72.1 337.3,72.1 343.4,72.1 349.5,72.1 355.6,72.1 361.8,72.1 367.9,72.1 374.0,72.1 380.1,72.1 386.2,72.1 392.4,72.1 398.5,72.1 404.6,72.1 410.7,72.1 416.8,72.1 422.9,72.1 429.1,72.1 435.2,72.1 441.3,72.1 447.4,72.1 453.5,72.1 459.6,72.1 465.8,72.1 471.9,89.0 478.0,89.0 484.1,89.0 490.2,89.0 496.4,89.0 502.5,89.0 508.6,89.0 514.7,89.0 520.8,89.0 526.9,89.0 533.1,72.1 539.2,72.1 545.3,72.1 551.4,72.1 557.5,72.1 563.6,72.1 569.8,72.1 575.9,72.1 582.0,72.1 588.1,84.7 594.2,84.7 600.4,101.7 606.5,101.7 612.6,101.7 618.7,101.7 624.8,101.7 630.9,101.7 637.1,101.7 643.2,101.7 649.3,89.0 655.4,89.0 661.5,72.1 667.6,72.1 673.8,72.1 679.9,72.1 686.0,72.1 692.1,72.1 698.2,72.1 704.4,72.1 710.5,72.1 716.6,72.1 722.7,72.1 728.8,72.1 734.9,72.1 741.1,72.1 747.2,72.1 753.3,72.1 759.4,72.1 765.5,72.1 771.6,72.1 777.8,72.1 783.9,72.1 790.0,72.1" fill="none" stroke="#c0392b" stroke-width="2.4"/><text x="798" y="76.1" font-size="10" fill="#c0392b">mean of 10</text><text x="62" y="32" font-size="10.5" fill="#68737f">the loop never sees this number — it only sees pass or fail</text></svg>', title=

Read the faint line as well as the solid one. The raw series stays jumpy to the
end, and that is not noise to be smoothed away — it is the search still trying
things. A loop whose picks stop varying has stopped exploring, which looks like
confidence and is indistinguishable from being stuck.

## What it learned about each candidate

The posterior is what the system believes, from outcomes only. Next to it, the
truth it was never told.

In [9]:
print(f"{'candidate':<18}{'runs':>6}{'believed':>10}{'true':>8}{'confidence':>12}")
for stage_id, pool in (("fetch", SOURCES), ("tidy", TIDIERS), ("check", CHECKS)):
    for candidate in pool:
        p = store.posterior(candidate)
        mark = "  <- best" if candidate == best_route.get(stage_id) else ""
        print(f"{candidate:<18}{p.runs:>6}{p.rate:>10.3f}{TRUTH[candidate]:>8.2f}"
              f"{p.confidence:>12.2f}{mark}")
    print()

candidate           runs  believed    true  confidence
fetch.alpha           13     0.267    0.35        0.62
fetch.beta            11     0.231    0.55        0.58
fetch.gamma           96     0.622    0.90        0.92  <- best

tidy.strict           13     0.267    0.55        0.62
tidy.loose            11     0.231    0.60        0.58
tidy.smart            96     0.622    0.88        0.92  <- best

check.shallow         13     0.267    0.80        0.62
check.deep            11     0.231    0.82        0.58  <- best
check.paranoid        96     0.622    0.78        0.92



Two things worth reading carefully.

The **believed** column tracks the truth in order, not in value. That is
expected and fine: a candidate is judged on whether the *step* worked, and the
search only needs the ordering to be right to pick correctly.

The **runs** column is uneven, and that is the loop working. Once a candidate
looks bad it gets tried less, so its count stops growing — but it is never cut
off entirely, because a candidate scores on what is known *plus* a bonus for
how little that is. Without that bonus the loop locks in: given a prior naming
the worst candidate best, a search on averages alone picked it sixty times out
of sixty and never tried the other two.

In [10]:
# explore=0 asks "what is the best you know", not "what should I try next".
# The loop above wanted the second question; this cell wants the first.
final = search.within(bench, bench.optimization_profiles[0],
                      evaluations=40, evidence=store, seed=999, explore=0.0)
print("what it would pick now:")
for stage_id, candidate in final.route.items():
    right = "correct" if candidate == best_route[stage_id] else \
        f"the best is {best_route[stage_id]}"
    print(f"  {stage_id:<8}{candidate:<16}{right}")
print(f"\ntrue quality of that route: {true_quality(final.route):.3f} "
      f"(best possible {true_quality(best_route):.3f})")

what it would pick now:
  fetch   fetch.gamma     correct
  tidy    tidy.smart      correct
  check   check.paranoid  the best is check.deep

true quality of that route: 0.618 (best possible 0.649)


## The check that keeps this honest

Picking each step on its own is only right while the steps are independent.
Nothing here guarantees that, so it is measured rather than assumed —
`interactions()` compares how pairs did together against how they did apart.

In [11]:
# `minimum` is how many times the pair must have run *together* before the
# comparison is allowed to say anything. The default of 3 is far too low here:
# three runs of a coin can look like anything, and the report fills with noise.
clashes = store.interactions(minimum=10)
if clashes:
    print("pairs that did worse together than apart:")
    for a, b, gap, runs in clashes[:5]:
        print(f"  {a:<16} + {b:<16} gap {gap:+.3f} over {runs} runs")
else:
    print("no pair did measurably worse together than apart —")
    print("which is what we would expect here, because the truth above really")
    print("is one number per candidate with no interaction built in.")

no pair did measurably worse together than apart —
which is what we would expect here, because the truth above really
is one number per candidate with no interaction built in.


Whatever it printed, read it as a *contrast*, not as a verdict on the pair. The
check compares routes containing both against routes containing exactly one of
them — same shape, differing only in whether the pair co-occurs.

That detail is the whole check. An earlier version compared the route outcome
against `rate(a) * rate(b)`, which is not like for like: a route succeeds only
if every step does, so a three-step route sits near 0.8³ = 0.51 while that
expectation was 0.8² = 0.64. Every pair looked like it clashed. On data built
with no interaction at all it reported eleven.

Two limits worth knowing.

A pair that *never* appears apart cannot be judged. With no contrast there is
no way to tell "this pair is bad" from "one of them is bad", and the honest
answer is to say nothing.

And `minimum` matters more than it looks. At the default of 3 this fills with
pairs that ran together three times and got unlucky. Ten is used below, and on
a job you cared about you would want more.

## What this notebook actually demonstrated

* A run produced a receipt.
* The receipt became evidence, keyed by **candidate** — not by stage, which
  would pool every option in a step under one belief and make the whole thing
  pointless.
* The next search started from that evidence, and the numbers written into the
  graph when it was drawn stopped mattering.
* The route it picks now is the best one there is, measured against a truth it
  was never shown.

Worth stating the cost as well. Optimism is what stops the loop locking on to a
bad prior, and it is not free: a search that only ever exploited reached a
decent route inside fifty runs here, where this one was still exploring. It
overtakes by about a hundred and then sits on the exact optimum. Faster to a
good answer, or slower to the best one — that is a real choice, and `explore`
is where you make it.

That is the argument this library makes, running rather than described.